# Цифровые технологии в профессиональной деятельности
## Раздел 1. Текст-майнинг
## Семинар 3. Культуромика и API НКРЯ

На прошлых семинарах мы освоили предобработку текста и провели мини-исследование на корпусе Посланий Президента.

Сегодня мы научимся работать с **Национальным корпусом русского языка** программно — через API. Это позволит нам воспроизвести подход А.А. Бонч-Осмоловской из статьи «Имена времени».

**План:**
1. Знакомимся с форматом JSON и учимся «раскапывать» вложенные структуры.
2. Подключаемся к API НКРЯ: статистика, портрет слова, конкордансы.
3. Воспроизводим исследование: конструкция «прилагательное + десятилетие».

In [ ]:
# Импорты
import requests
import json
import time
import pandas as pd
import matplotlib.pyplot as plt

## Подготовительная часть: Google N-gram Viewer

Google N-gram Viewer: https://books.google.com/ngrams/

Предложите запрос для сервиса, попробуйте подобрать что-нибудь интересное.


---
## Часть 1. Подключение к API Национального корпуса русского языка

### Что такое API

**API (Application Programming Interface)** — это набор правил, по которым одна программа может общаться с другой. Когда вы открываете НКРЯ в браузере и вводите запрос, браузер отправляет HTTP-запрос на сервер. Сервер обрабатывает его и возвращает ответ — список найденных контекстов. Мы можем делать то же самое из Python с помощью библиотеки `requests`, минуя браузер.

Всё это время вы были знакомы с API, каждый день пользуетесь им через графический интерфейс браузера: вы гуглите информацию. Давайте для примера что-нибудь загуглим и посмотрим в адресную строку.

### 1.1. Авторизация


In [ ]:
# Вставьте свой токен ниже:
API_TOKEN = ""  # <-- вставьте свой токен сюда

# !!! НЕ публикуйте токен на GitHub !!!
# Заголовок авторизации — он будет добавляться к каждому запросу
HEADERS = {"Authorization": f"Bearer {API_TOKEN}"}

# Базовый URL — все запросы к API начинаются с этого адреса
BASE_URL = "https://ruscorpora.ru/api/v1"

if not API_TOKEN:
    print("⚠ Вставьте свой API-токен в переменную API_TOKEN.")
else:
    print(f"Токен настроен ({API_TOKEN[:8]}...)")

### 1.2. JSON — формат обмена данными


In [ ]:
# Пример: вложенный JSON (словарь Python)
example = {
    "student": {
        "name": "Алиса",
        "courses": ["история", "философия"],
        "grades": {"история": 9, "философия": 8}
    }
}

# Как добраться до оценки по истории?
# Шаг за шагом: словарь -> ключ -> ключ -> ключ
print(example["student"])                    # весь словарь студента
print(example["student"]["name"])            # "Алиса"
print(example["student"]["courses"][0])      # "история" (первый элемент списка)
print(example["student"]["grades"]["история"])  # 9

В Python для работы с JSON используется модуль `json`:


### 1.3. Первый запрос: статистика корпуса


In [ ]:
# import json
# import requests

# Эндпоинт — это конкретный URL-адрес (например, /api/users), по которому приложение (клиент) 
# обращается к серверу для получения или отправки данных через API.

# Эндпоинт НКРЯ /stats/ не требует сложных параметров.

# Параметры запроса передаются как JSON-строка.
# {"type": "MAIN"} — основной корпус НКРЯ.
# Другие варианты: "PAPER" (газетный), "PARA" (параллельный) и др.

params = {"corpus": json.dumps({"type": "MAIN"})}

# requests.get() отправляет HTTP GET-запрос.
# Три аргумента:
#   1) URL — адрес эндпоинта
#   2) params= — параметры, которые добавятся к URL после знака ?
#                   (Символ ?: Отделяет URL от параметров)
#   3) headers= — заголовки запроса (у нас там токен авторизации)
response = requests.get(f"{BASE_URL}/stats/", 
                        params=params, 
                        headers=HEADERS)

# response.status_code — числовой код ответа сервера:
#   200 = всё хорошо, данные получены
#   401 = не авторизован (проверьте токен)
#   400 = ошибка в параметрах запроса
#   500 = ошибка на стороне сервера
print(f"Статус ответа: {response.status_code}")

# response.json() — превращаем текст ответа обратно в словарь Python.
stats = response.json()

# json.dumps(..., indent=2) — красиво печатаем с отступами.
# ensure_ascii=False — чтобы кириллица отображалась нормально, а не как \u0410.
print(json.dumps(stats, indent=2, ensure_ascii=False))

### 1.4. Портрет слова


In [ ]:
# Здесь параметр называется query (а не corpus) и содержит больше полей.
query = json.dumps({
    "lemma": "дорога",                   # лемма — начальная форма слова
    "corpus": {"type": "MAIN"},          # в каком корпусе искать
    "seed": 12345,                       # число для воспроизводимости (как random seed)
    "resultType": ["PORTRAIT_WORD_INFO"] # какой тип информации вернуть
})

# Обратите внимание: params={"query": query} — мы передаём JSON-строку
# как значение параметра query в URL.
response = requests.get(
    f"{BASE_URL}/word-portrait/",
    params={"query": query},
    headers=HEADERS
)
print(f"Статус: {response.status_code}")

portrait = response.json()

print(json.dumps(portrait, indent=2, ensure_ascii=False))

In [ ]:
portrait # итак посмотрим снова на этот ~~кровавый кошмар~~ ответ сервера

# на самом деле он проще, чем кажется

In [ ]:
type(portrait) # наш ответ является словарём, поэтому мы можем легко обращаться к значениям по ключам

In [ ]:
for key in portrait:   # а какие ключи у нас вообще есть?
    print(key)

`possiblePos` — какая часть речи предполагается


In [ ]:
portrait['possiblePos'] # например, какая часть речи предполагается?

In [ ]:
portrait['propsData'] # посмотрим на словарь данных

In [ ]:
portrait['propsData']['items'] # заглянём в список словарей-разборов

In [ ]:
portrait['propsData']['items']['parsingFields'] # откроем-ка разбор...

In [ ]:
# нам выдало ошибку, потому что `portrait['propsData']['items']` - список
# значит, можем пробежаться по нему циклом

for item in portrait['propsData']['items']:
    print(item)

# возьмём первое значение в этом списке, посмотрим, что там за элемент
type(portrait['propsData']['items'][0])

In [ ]:
# снова в нашем распоряжении словарь, снова пробежимся по ключам: оказывается, что ключ один

for item in portrait['propsData']['items'][0]:
    print(item)

In [ ]:
# итак, у нас есть список словарей, возьмём первый элемент списка, а потом ключом откроем дверцу к заветным разборам

for item in portrait['propsData']['items'][0]['parsingFields']:
    print(item)

In [ ]:
# Давайте напишем небольшой парсер на основе выдачи запроса

def parse_portrait(portrait):

    options = {}
    for i, block in enumerate(portrait['propsData']['items'], 1):
        razbor = {}

        for item in block['parsingFields']:
            name = item["name"]
            
            values = []
            for value in item['value']:
                values.append(value['valString']['v'])

            razbor[name] = values
            options[i] = razbor
    
    return options

In [ ]:
parsed_items = parse_portrait(portrait)

parsed_items # в таком виде уже как-то поприятнее, без всяких лишних тегов

#### Задание 1

Напишите запрос к Портрету слова НКРЯ для любого слова на ваш выбор. Выведите все разборы через `parse_portrait`.

**Дополнительно (посложнее):** выберите слово с несколькими частями речи (например, *печь*). Сгруппируйте разборы по частям речи и выведите их отдельными списками.

In [ ]:
# ВАШ КОД ЗАПРОСА

In [ ]:
# ВАШ КОД ГРУППИРОВКИ

## Часть 2. Поиск конкордансов


In [ ]:
# Структура запроса вложенная (4 уровня!):
# lexGramm -> sectionValues -> subsectionValues -> conditionValues
#
# sectionValues  — массив «блоков поиска» (обычно один)
# subsectionValues — массив слов в запросе (одно слово = один элемент)
# conditionValues  — условия для конкретного слова

query = json.dumps({
    "corpus": {"type": "MAIN"},
    "lexGramm": {
        "sectionValues": [{                  # один блок поиска
            
            "subsectionValues": [{           # одно слово
                
                "conditionValues": [{        # одно условие
                    
                    "fieldName": "lex",      # lex = искать по лемме
                    "text": {"v": "дорога"}  # значение леммы
                }]
            }]
        }]
    },
    "params": {
        "pageParams": {
            "page": 0,              # страница результатов (нумерация с 0)
            "snippetsPerPage": 5    # сколько контекстов вернуть
        },
        "seed": 42  # для воспроизводимости порядка результатов
    }
})


response = requests.get(
    f"{BASE_URL}/lex-gramm/concordance",
    params={"query": query},
    headers=HEADERS
)
print(f"Статус: {response.status_code}")

data = response.json()

# Смотрим, какие ключи есть в ответе
print(f"Ключи в ответе: {list(data.keys())}")

In [ ]:
# Посмотрим на сырой JSON — с чем нам предстоит работать
print(json.dumps(data, indent=2, ensure_ascii=False)[:2000])

### 2.1. Разбираем JSON-ответ конкорданса


In [ ]:
# Посмотрим, как это выглядит на практике.
# Выведем текст и подсвеченные слова из первых контекстов.

for group in data.get('groups', []):
    for doc in group.get('docs', [])[:3]:  # берём первые 3 документа
        # Название текста — самый простой путь к метаданным
        title = doc['info']['title']
        print(f"Название документа: '{title}'\n")

In [ ]:
for group in data.get('groups', []):
    for doc in group.get('docs', [])[:3]:  # берём первые 3 документа
        # Название текста — самый простой путь к метаданным
        title = doc['info']['title']
        
        # Внутри документа может быть несколько сниппетов (контекстов)
        for sg in doc.get('snippetGroups', []):
            for snippet in sg.get('snippets', []):
                for seq in snippet.get('sequences', []):
                    words = seq.get('words', [])

                    print(words)

In [ ]:
for group in data.get('groups', []):
    for doc in group.get('docs', [])[:3]:  # берём первые 3 документа
        # Название текста — самый простой путь к метаданным
        title = doc['info']['title']
        
        # Внутри документа может быть несколько сниппетов (контекстов)
        for sg in doc.get('snippetGroups', []):
            for snippet in sg.get('snippets', []):
                for seq in snippet.get('sequences', []):
                    words = seq.get('words', [])
                    
                    # Склеиваем текст: каждый элемент words содержит
                    # либо слово (type=WORD), либо пробел/пунктуацию (type=PLAIN)
                    full_text = ''.join(w['text'] for w in words)
                    print(full_text, '\n')

In [ ]:
for group in data.get('groups', []):
    for doc in group.get('docs', [])[:3]:  # берём первые 3 документа
        # Название текста — самый простой путь к метаданным
        title = doc['info']['title']
        
        # Внутри документа может быть несколько сниппетов (контекстов)
        for sg in doc.get('snippetGroups', []):
            for snippet in sg.get('snippets', []):
                for seq in snippet.get('sequences', []):
                    words = seq.get('words', [])
                    
                    # Склеиваем текст: каждый элемент words содержит
                    # либо слово (type=WORD), либо пробел/пунктуацию (type=PLAIN)
                    full_text = ''.join(w['text'] for w in words)
                    print(full_text,)
                    
                    # Подсвеченные слова — те, что попали в запрос.
                    # У них displayParams — непустой словарь.
                    highlighted = [w['text'] for w in words
                                   if w.get('displayParams') != {}]
                    
                    print(highlighted, '\n')

In [ ]:
# Итоговый результат

for group in data.get('groups', []):
    for doc in group.get('docs', [])[:3]:  # берём первые 3 документа
        # Название текста — самый простой путь к метаданным
        title = doc['info']['title']
        
        # Внутри документа может быть несколько сниппетов (контекстов)
        for sg in doc.get('snippetGroups', []):
            for snippet in sg.get('snippets', []):
                for seq in snippet.get('sequences', []):
                    words = seq.get('words', [])
                    
                    # Склеиваем текст: каждый элемент words содержит
                    # либо слово (type=WORD), либо пробел/пунктуацию (type=PLAIN)
                    full_text = ''.join(w['text'] for w in words)
                    
                    # Подсвеченные слова — те, что попали в запрос.
                    # У них displayParams — непустой словарь.
                    highlighted = [w['text'] for w in words
                                   if w.get('displayParams') != {}]
                    
                    if len(full_text) > 20:  # пропускаем пустые сниппеты
                        print(f"{title}")
                        print(f"{full_text[:200]}")
                        if highlighted:
                            print(f"Найдено: {highlighted}")
                        print()

### 2.2. Извлекаем метаданные


In [ ]:
# Метаданные (автор, год, жанр) хранятся в странной структуре:
# doc["info"]["docExplainInfo"]["items"][0]["parsingFields"]
# Это список словарей вида: {"name": "author", "value": [{"valString": {"v": "Пушкин"}}]}
#
# Напишем функцию, которая превращает это в обычный словарь {"author": "Пушкин"}

def extract_metadata(doc):
    """Извлекает метаданные документа из parsingFields в плоский словарь."""
    metadata = {}
    try:
        # Добираемся до списка полей
        fields = doc['info']['docExplainInfo']['items'][0]['parsingFields']
        for field in fields:
            name = field['name']  # например, "author", "created", "sphere"

            # Значение закопано глубоко: value -> первый элемент -> valString -> v
            value = field['value'][0]['valString']['v']
            metadata[name] = value

    except (KeyError, IndexError):
        # Если какого-то поля нет — не падаем, просто пропускаем
        pass
    return metadata

In [ ]:
# Проверим на первом документе из предыдущего запроса
first_doc = data['groups'][0]['docs'][0]
meta = extract_metadata(first_doc)
print("Метаданные первого документа:")
for k, v in meta.items():
    print(f"  {k}: {v}")

### 2.3. Собираем всё в DataFrame


In [ ]:
# import pandas as pd

# Объединяем всё: из каждого документа извлекаем текст контекста,
# подсвеченные слова и метаданные. Результат — список словарей -> DataFrame.

def parse_concordance(data):
    """Парсит JSON-ответ конкорданса в список словарей для DataFrame."""
    rows = []
    
    # Уровень 1: groups — обычно один элемент
    for group in data.get('groups', []):
        
        # Уровень 2: docs — каждый документ = один текст из корпуса
        for doc in group.get('docs', []):
            title = doc['info']['title']
            meta = extract_metadata(doc)  # автор, год, жанр...
            
            # Уровень 3: snippetGroups -> snippets -> sequences -> words
            for sg in doc.get('snippetGroups', []):
                for snippet in sg.get('snippets', []):
                    for seq in snippet.get('sequences', []):
                        words = seq.get('words', [])
                        
                        # Текст контекста
                        full_text = ''.join(w['text'] for w in words)
                        
                        # Слова, попавшие в запрос
                        highlighted = [w['text'] for w in words
                                      if w.get('displayParams') != {}]
                        
                        # Фильтруем слишком короткие (мусор)
                        if len(full_text) > 10:
                            rows.append({
                                'text': full_text.strip(),
                                'highlighted': ', '.join(highlighted),
                                'title': title,
                                'author': meta.get('author', ''),
                                'year': meta.get('created', ''),
                                'sphere': meta.get('sphere', ''),
                            })
    return rows

In [ ]:
# Парсим ответ предыдущего запроса (лемма "дорога")
rows = parse_concordance(data)

df_conc = pd.DataFrame(rows)

print(f"Извлечено {len(df_conc)} контекстов")
df_conc.head(10)

Мы научились получать выдачу конкорданса и засовывать её в датафрейм. Осталось научиться считывать информацию с нескольких страниц.

In [ ]:
# Функция для скачивания всех страниц конкорданса.
# API возвращает результаты постранично (как поисковик).
# Эта функция перебирает страницы, пока не закончатся результаты.

import time

def get_full_concordance(query_str, max_pages=10):
    """
    Скачивает все страницы конкорданса и возвращает DataFrame.
    
    query_str — JSON-строка запроса (результат json.dumps)
    max_pages — максимум страниц (защита от бесконечного цикла)
    """
    all_rows = []

    for page in range(max_pages):
        # Меняем номер страницы в запросе
        query_dict = json.loads(query_str)
        query_dict["params"]["pageParams"]["page"] = page

        response = requests.get(
            f"{BASE_URL}/lex-gramm/concordance",
            params={"query": json.dumps(query_dict)},
            headers=HEADERS
        )
        
        if response.status_code != 200:
            print(f"  Страница {page}: ошибка {response.status_code}")
            break

        page_data = response.json()
        rows = parse_concordance(page_data)

        if not rows:
            break  # пустая страница — результаты закончились

        all_rows.extend(rows)
        time.sleep(0.5)  # пауза между страницами

    return pd.DataFrame(all_rows)

In [ ]:
# Скачиваем все страницы для запроса "дорога"
df_rnc = get_full_concordance(query, max_pages=10)
print(f"Скачано {len(df_rnc)} контекстов")

In [ ]:
print(len(df_rnc))
df_rnc.head()

## Часть 3. Конструкция из двух слов: «прилагательное + десятилетие»


In [ ]:
# В subsectionValues указываем ДВА элемента:
#   1) первое слово — прилагательное (условие: часть речи A)
#   2) второе слово — лемма «дорога» (условие: lex + расстояние dist)
#
# dist = {"begin": -1, "end": 1} означает: второе слово может стоять
# на 1 позицию левее или правее первого (т.е. рядом, в любом порядке)

query_two_words = json.dumps({
    "corpus": {"type": "MAIN"},
    "lexGramm": {
        "sectionValues": [{
            "subsectionValues": [
                {
                    # --- Первое слово: любое прилагательное ---
                    "conditionValues": [{
                        "fieldName": "gramm",
                        "text": {"v": "A"}
                    }]
                },
                {
                    # --- Второе слово: лемма «дорога» ---
                    "conditionValues": [
                        {
                            "fieldName": "lex",
                            "text": {"v": "дорога"}
                        },
                        {
                            # Расстояние от первого слова: от -1 до 1
                            "fieldName": "dist",
                            "intRange": {"begin": -1, "end": 1}
                        }
                    ]
                }
            ]
        }]
    },
    "params": {
        "pageParams": {"page": 0, "snippetsPerPage": 50},
        "seed": 42
    }
})

In [ ]:
# Скачиваем все страницы для конструкции «прилагательное + дорога»
df_road = get_full_concordance(query_str=query_two_words, max_pages=10)

In [ ]:
print(f"Извлечено контекстов: {len(df_road)}")
print(f"Колонки датафрейма: {df_road.columns}")

In [ ]:
# Выводим таблицу: подсвеченные слова, текст, автор, год
df_road.head(15)

### 3.1. Воспроизводим исследование: цикл по десятилетиям


In [ ]:
# Словарь: метка -> лемма порядкового числительного
decades = {
    '20-е': 'двадцатый',
    '30-е': 'тридцатый',
    '40-е': 'сороковой',
    '50-е': 'пятидесятый',
    '60-е': 'шестидесятый',
    '70-е': 'семидесятый',
    '80-е': 'восьмидесятый',
    '90-е': 'девяностый'
}

def make_adj_decade_query(decade_lemma, page=0, per_page=10):
    """Формирует JSON-запрос: прилагательное + десятилетие (мн. ч.)"""
    return json.dumps({
        "corpus": {"type": "MAIN"},
        "lexGramm": {
            "sectionValues": [{
                "subsectionValues": [

                    # Первое слово: прилагательное
                    {"conditionValues": [
                        {"fieldName": "gramm", "text": {"v": "A"}}
                    ]},

                    # Второе слово: десятилетие (лемма + мн. число + расстояние)
                    {"conditionValues": [
                        {"fieldName": "lex", "text": {"v": decade_lemma}},
                        {"fieldName": "gramm", "text": {"v": "pl"}},
                        {"fieldName": "dist", "intRange": {"begin": -1, "end": 1}}
                    ]}
                ]
            }]
        },
        "params": {
            "pageParams": {"page": page, "snippetsPerPage": per_page},
            "seed": 42
        }
    })

In [ ]:
print("Функция готова. Пример запроса для 90-х:")
make_adj_decade_query("девяностый")

In [ ]:
import time

decade_stats = {}
all_decade_rows = []

for label, lemma in decades.items():
    q = make_adj_decade_query(lemma, per_page=50)
    
    # Получаем wordUsageCount из первого запроса
    resp = requests.get(
        f"{BASE_URL}/lex-gramm/concordance",
        params={"query": q},
        headers=HEADERS
    )
    if resp.status_code == 200:
        total = int(resp.json().get('queryStats', {}).get('wordUsageCount', 0))
        decade_stats[label] = total
    
    # Скачиваем все страницы
    df_one = get_full_concordance(query_str=q, max_pages=10)
    df_one['decade'] = label
    all_decade_rows.append(df_one)
    
    print(f"{label} ({lemma}): {total} вхождений, извлечено {len(df_one)} контекстов")
    time.sleep(1)

df_decades = pd.concat(all_decade_rows, ignore_index=True)
print(f"\nИтого: {len(df_decades)} контекстов по {len(decades)} десятилетиям")

In [ ]:
print(f"Размер таблицы: {df_decades.shape}")
print(f"Десятилетия: {df_decades['decade'].value_counts().to_dict()}")

In [ ]:
df_decades[['decade', 'highlighted', 'text', 'author', 'year']].head(15)

### 3.2. Сравниваем десятилетия


In [ ]:
# Воспроизводим Рис. 1 из статьи Бонч-Осмоловской:
# столбчатая диаграмма частотности конструкции по десятилетиям.

labels = list(decade_stats.keys())    # ['20-е', '30-е', ..., '90-е']
values = list(decade_stats.values())  # [35, 38, ..., 57]
mean_val = sum(values) / len(values)  # среднее значение

plt.figure(figsize=(10, 5))

# plt.bar() — столбчатая диаграмма
bars = plt.bar(labels, values, color='steelblue', edgecolor='navy')

# plt.axhline() — горизонтальная линия на уровне среднего.
# y= — координата по оси Y
# linestyle='--' — пунктир
plt.axhline(y=mean_val, color='red', linestyle='--', label=f'Среднее: {mean_val:.0f}')

plt.title('Частотность конструкции «прилагательное + десятилетие» в НКРЯ', fontsize=14)
plt.ylabel('Число вхождений', fontsize=12)
plt.legend()

# Подписи значений над столбцами
for i, v in enumerate(values):
    plt.text(i, v + 1, str(v), ha='center', fontweight='bold')

plt.show()

#### Задание 2

Придумайте свою конструкцию из двух слов (например, «прилагательное + местоимение», «наречие + говорить» или что угодно). Сформируйте запрос, получите конкордансы, выведите в DataFrame. Отправьте результат в телеграм.

In [ ]:
# ВАШ КОД
